In [ ]:
!pip install bs4 requests
!pip install openai

In [ ]:
from bs4 import BeautifulSoup, NavigableString
import requests
import re
import os
from collections import deque
import json

In [ ]:
#storing links that have been accessed, valid or not
SAVE_DIR = r"../raw_data"
EXCLUDED_PATH = os.path.join(SAVE_DIR, "excluded_links.json")
DEPTH_LIMIT = 5

In [ ]:
keywords = [
    "ca sĩ việt nam", "nam ca sĩ việt nam", "nữ ca sĩ việt nam",
    "ca sĩ gốc việt", "ca sĩ hải ngoại", "nhạc sĩ việt nam",
    "ban nhạc việt nam", "ban nhạc rock việt nam",
    "nhà sản xuất thu âm việt nam", "nhà sản xuất âm nhạc việt nam",
    "nhạc sĩ hòa âm phối khí việt nam", "rapper việt nam" 
]

In [ ]:
#wikipedia requires a header to access its content.
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

In [ ]:
#seed for accessing data
SEED = "https://vi.wikipedia.org/wiki/S%C6%A1n_T%C3%B9ng_M-TP"

In [ ]:
response = requests.get(SEED, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
whole_page = soup.find('div', class_='mw-content-ltr')

In [ ]:
print(whole_page)

In [ ]:
#lấy data từ phần tiểu sử 
paragraph_lists = whole_page.find_all('p')
print(paragraph_lists)

In [ ]:
relations = []

In [ ]:
for idx, p in enumerate(paragraph_lists):
    print(f"\n===== 🟦 Đang xử lý đoạn <p> thứ {idx} =====")

    p_html = str(p)
    a_tag_lists = p.find_all('a')

    if not a_tag_lists:
        print("⛔ Không có thẻ <a> trong đoạn này.")
        continue

    for a_tag in a_tag_lists:
        href = a_tag.get('href')
        anchor_html = str(a_tag)

        if not href:
            print("⚠️ Thẻ <a> không có href -> bỏ qua.")
            continue

        # Chuẩn hóa link
        link = "https://vi.wikipedia.org" + href
        print(f"\n🔗 Đang kiểm tra link: {link}")

        # Tìm vị trí của thẻ <a> trong HTML gốc
        pos = p_html.find(anchor_html)
        if pos != -1:
            start = max(0, pos - 100)
            end   = min(len(p_html), pos + len(anchor_html) + 100)
            surrounding_text = p_html[start:end]

            short_preview = surrounding_text.replace("\n", " ")
            print(f"   📎 Text xung quanh thẻ <a>: …{short_preview[:120]}…")

        else:
            surrounding_text = ""
            print("   ⚠️ Không tìm thấy anchor_html trong đoạn <p>!")

        # Gọi request để kiểm tra category của trang
        try:
            print("   🌐 Gửi request…")
            sub_resp = requests.get(link, headers=headers, timeout=6)
            sub_soup = BeautifulSoup(sub_resp.text, 'html.parser')
            cat_div = sub_soup.find('div', id='mw-normal-catlinks')

            if not cat_div:
                print("   ⚠️ Không tìm thấy category trong trang này.")
                continue

            cat_text = cat_div.get_text(strip=True).lower()
            print(f"   📂 Category: {cat_text}")

            # Kiểm tra keyword
            if any(k in cat_text for k in keywords):
                print("   ✅ Phù hợp keyword! → THÊM relation")
                relation = {
                    'link': link,
                    'content': surrounding_text
                }
                relations.append(relation)
                print("   ➤ relation:", relation)
            else:
                print("   ❌ Không khớp keyword → bỏ qua")

        except Exception as e:
            print(f"   ❗ Lỗi khi truy cập {link}: {e}")


In [ ]:
print(relations)

In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
def evaluate_relations(relation):
    response = client.responses.create(
        model="gpt-5-nano",
        reasoning={"effort": "low"},
        input=[
            {'role': 'developer',
            'content': "Evaluate the relation provided by user in 'content' field and return its category. The category includes 'relative' for family/love relation, 'former_member' if the singer used to be a member of the band, 'collaborated_with' if the singer with target has collaborated in a song/project/movie, 'inspired_by' if the singer is inspired by the target, 'inspiration_of' if the singer inspires the target. 'Unknown' if you can decide the category"},
            {'role': 'user',
             'content': f"Evaluate this relation: {relation}"
            }
            ]
    )
    text = response.output_text
    if 'Reasoning:' in text:
        category = text.split('Reasoning:')[0].strip()
    relation['category'] = text
    return relation

In [ ]:
print(relations[0], '\n')
print(relations[1], '\n')
print(relations[2], '\n')
print(relations[3], '\n')
for i in range (0, 4):
    print(evaluate_relations(relations[i]))